# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Skill loaded for this assignment: `skills/framing-ml-problems/SKILL.md`.*

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring (provisional, confirmable through Week 4).**

I'm choosing this lane as my starting point because the starter repo already ships a working, runnable end-to-end example of it (`scripts/01`–`05`), so I can learn the full workflow — prepare features, build a transparent baseline, train a model, evaluate, export — by reading and running real code before I try to extend it myself. It also produces the clearest story for a beginner: one row is one page, the output is a ranked list, and there's an obvious real-world action (a reviewer opens the top page first) with an obvious cost if the ranking is wrong (wasted reviewer time, or a genuinely declining page left unreviewed). I'll confirm or swap this lane by the end of Week 4 once I've done more EDA.

*(That paragraph is why this lane is a good place for **me to learn**. Section 2 answers the separate question of why the **project itself** deserves ML — the skill's fourth framing question.)*

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Which pages, out of a client's content inventory, should a content reviewer look at first for a possible refresh?

**Unit of analysis:** one row = one content page (`content_id`), aggregated over its trailing 90-day window.

**Decision improved:** where a content team spends its limited review time this cycle — today that decision is either ad-hoc or based on a hand-built rule; this project tests whether a ranked, evidence-backed queue does better.

**Output:** a ranked list (a "review queue") of pages, each with a score and reason codes explaining *why* it's in the queue (e.g. "declining_with_demand", "stale_visible_page").

**Action someone takes:** a content reviewer opens the top N pages on the list first and decides whether to refresh, expand, or leave each one — the model/score never edits or publishes anything itself; a human always makes the final call.

### 2.1 Cost of a wrong call — two directions, not symmetric

- *False positive* (page ranked high but didn't actually need review): a reviewer spends ~10–15 minutes checking a page that turns out fine. Annoying, **bounded**, and self-correcting — the reviewer discovers the mistake immediately.
- *False negative* (a genuinely declining page never makes the queue): the page keeps losing visibility/traffic for weeks or months. The cost is **unbounded and invisible** — it grows every week, and nobody knows it is growing, because the page simply isn't on the list for anyone to look at.

The asymmetry is not just "one is bigger". It is structural: one error is capped and observed, the other compounds and is unobserved. That is what makes the false negative the expensive side.

**How that asymmetry gets acted on:** because this is a *ranking* task (see 2.3), the knob is **K — how many pages the reviewer opens** — not a probability cut-off. Raising K catches more true declines at the price of more reviewer hours; lowering it saves hours and lets more declines slip through silently. I'll pick K from the reviewer's real capacity first, then ask whether widening it is worth the hours.

### 2.2 Why data/ML at all, and not a plain rule?

This project does not have to *assume* a hand-written rule is insufficient — the rule already exists and is already measured. `scripts/02_baseline_score.py` implements a transparent hand rule, and the repo documents its result on this slice: **Precision@50 ≈ 0.24**, against **≈ 0.68–0.74** for the learned model (the ~3x lift is the stable claim; the third decimal is library-version sensitive).

So the honest answer to "why ML?" is: the pattern appears to be real but spread across several tangled signals with non-obvious weights — exactly the situation the framing skill describes as earning ML its place. If the two numbers had come out close, the hand rule would be the better answer: cheaper, more transparent, no retraining, and it explains itself.

**I keep both anyway.** The baseline produces human-readable reason codes that a model does not; `04_evaluate_and_export.py` blends them, so the model supplies the ordering and the rule supplies the explanation a reviewer actually reads.

### 2.3 Task type and target

**Task type: ranking / scoring**, not classification. The question is "which ones first?", and the framing skill maps that to a priority score measured by precision@K.

The *implementation* still runs through a classifier, and it is worth being explicit about why that is not a contradiction: the only label available is binary (`is_declining_label`), so `03_train_model.py` trains classifiers and I use each page's **predicted probability as the priority score** to rank by.

```
ranking question  →  trained as classification (binary label)
                  →  outputs a probability per page
                  →  probability used as the ranking score
                  →  scored with Precision@K
```

Two consequences I want on the record now:
- **Accuracy is the wrong metric here.** Accuracy grades 30,000 separate yes/no verdicts. The reviewer makes one decision: which K pages to open.
- **There is no probability threshold in this design.** The cut is positional (top K), not a value cut-off. Moving a 0.5 cut-off to 0.3 would change how many pages are *labelled* declining and change nothing at all about which pages get opened, because the ordering is unaffected.

### 2.4 The target is a proxy, and it is a rule-defined one

The framing skill states the rule plainly: *the target must be observed, not defined* — a label that comes from someone's rule means the model learns the rule, not the world.

Measured against that, my target does not fully pass, and I'd rather say so than let a reviewer find it:

```
is_declining_label = (trend_direction == "down")
```

- **What I actually care about:** "this page is worth a reviewer's time."
- **What I can measure:** "this page fell, according to the `trend_direction` rule."

Those are not the same thing, and the gap runs in **both directions**: a healthy page that dipped for seasonal reasons gets rewarded into the queue, while a genuinely weak page that has been flat at the bottom for a year never appears. So every accuracy number in this project is accuracy *against the proxy*, not against the truth.

The cleaner target would be an outcome measured in a **later** time window (e.g. does the page's performance in the next 90 days actually deteriorate?). I am noting that as a candidate improvement for the capstone rather than pretending the current label is an observed outcome.

### 2.5 Metric, named before training

Per the skill: *name the metric before training — "good" defined after the fact always looks good.* So, on the record:

- **Primary metric: Precision@50.** It measures the decision that is actually taken ("open the top 50"). K = 50 is a placeholder for one review cycle's realistic capacity; if the real capacity turns out to be 20 or 200, K changes and the evaluation is redone. K should follow the reviewer, not the repo.
- **Always reported beside two other numbers**, because Precision@K alone can flatter: (1) the **base rate** on the same split — if 70% of pages are already labelled declining, a random draw of 50 scores ≈ 0.70 and a 0.74 model has gained almost nothing; and (2) the **hand-rule baseline** on the *same* split. Comparisons across different splits don't count.
- **ROC-AUC: secondary, for diagnosis only — not for choosing between models.** AUC is threshold-free and base-rate-independent, which makes it useful for reading a failure: high AUC with low Precision@50 means the model learned something but the *top* of the list is bad; low on both means it didn't learn. But AUC weights a #29,000-vs-#29,500 comparison the same as #3-vs-#700, and no human ever sees the former. Writing this down now is what stops me from quietly switching to AUC later if it happens to look nicer.

### 2.6 The one-paragraph frame

> For a **FlyRank content reviewer**, deciding **which pages to open first for a possible refresh this cycle**, we will build a **ranked review queue with scores and reason codes** from the **anonymized 90-day content-performance slice**, scoring **each page's probability of carrying the `is_declining_label` proxy**, measured by **Precision@50 (reported beside the base rate and the hand-rule baseline, on a client-held-out split)**. A wrong call costs **~10–15 minutes of reviewer time in one direction, and weeks of silent, compounding decline in the other**. A plain rule isn't enough because **the hand rule is already built and measured at Precision@50 ≈ 0.24, roughly a third of the learned model** — the signal is real but spread across tangled features. We will claim only **observed / directional / decision-support** results.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Every number below is printed by the cell, not typed by me — so the prose underneath can never drift away from the data.

In [1]:
import pandas as pd

# Load the starter dataset (relative path from work/notebooks/ up to repo root)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

n_rows, n_cols = df.shape
n_clients = df["client_id"].nunique()

print("=" * 62)
print("1) HOW BIG IS THE INVENTORY WE WOULD BE RANKING?")
print("=" * 62)
print(f"Rows (pages)      : {n_rows:,}")
print(f"Columns           : {n_cols}")
print(f"Distinct clients  : {n_clients}")
print(f"Median pages/client: {df.groupby('client_id').size().median():.0f}")

1) HOW BIG IS THE INVENTORY WE WOULD BE RANKING?
Rows (pages)      : 30,000
Columns           : 44
Distinct clients  : 32
Median pages/client: 567


In [2]:
# 2) BASE RATE -- the single most important number in this notebook.
#    trend_direction == 'down' is the exact rule the label is built from.
#    Reading it here to DESCRIBE the data; never using it as a feature.

print("=" * 62)
print("2) BASE RATE (the floor any model must beat)")
print("=" * 62)

counts = df["trend_direction"].value_counts(dropna=False)
print("trend_direction values:")
for value, count in counts.items():
    print(f"   {str(value):<12} {count:>7,}  ({count / n_rows:.1%})")

base_rate = (df["trend_direction"] == "down").mean()
print(f"\nBase rate (share labelled 'down'): {base_rate:.1%}")
print(f"-> A RANDOM pick of 50 pages would score Precision@50 ~= {base_rate:.3f}")
print(f"-> Documented hand-rule baseline    : Precision@50  = 0.240")
print(f"-> Documented learned model         : Precision@50 ~= 0.680-0.740")

print("\nHeadroom check:")
print(f"   model - random   : {0.74 - base_rate:+.3f}")
print(f"   model - hand rule: {0.74 - 0.240:+.3f}")
if base_rate > 0.24:
    print("   NOTE: the hand rule scores BELOW a random draw at this base rate.")
    print("         Worth flagging in the report -- it is a finding, not a bug.")

2) BASE RATE (the floor any model must beat)
trend_direction values:
   down          16,262  (54.2%)
   stable         5,962  (19.9%)
   up             4,388  (14.6%)
   new            2,236  (7.5%)
   flat           1,152  (3.8%)

Base rate (share labelled 'down'): 54.2%
-> A RANDOM pick of 50 pages would score Precision@50 ~= 0.542
-> Documented hand-rule baseline    : Precision@50  = 0.240
-> Documented learned model         : Precision@50 ~= 0.680-0.740

Headroom check:
   model - random   : +0.198
   model - hand rule: +0.500
   NOTE: the hand rule scores BELOW a random draw at this base rate.
         Worth flagging in the report -- it is a finding, not a bug.


In [3]:
# 3) Are the declining pages actually worth reviewing -- is there real demand?
#    (a page declining from 0 impressions is not a useful queue candidate)

print("=" * 62)
print("3) ARE THE 'DOWN' PAGES WORTH A REVIEWER'S TIME?")
print("=" * 62)

declining = df["trend_direction"] == "down"
visible   = df["impressions_90d"] >= 100
candidates = df[declining & visible]

print(f"Declining pages                      : {declining.sum():,}")
print(f"...of those, with >=100 impressions  : {len(candidates):,} "
      f"({len(candidates) / max(declining.sum(), 1):.1%} of declining)")
print(f"...as a share of the whole dataset   : {len(candidates) / n_rows:.1%}")
print(f"\nImpressions among real candidates:")
print(f"   median : {candidates['impressions_90d'].median():,.0f}")
print(f"   p90    : {candidates['impressions_90d'].quantile(0.90):,.0f}")

reviewable_cycles = len(candidates) / 50
print(f"\nAt K=50 per cycle, this candidate pool is ~{reviewable_cycles:,.0f} review cycles")
print("-> prioritisation is the problem; the pool is far larger than capacity.")

3) ARE THE 'DOWN' PAGES WORTH A REVIEWER'S TIME?
Declining pages                      : 16,262
...of those, with >=100 impressions  : 13,152 (80.9% of declining)
...as a share of the whole dataset   : 43.8%

Impressions among real candidates:
   median : 1,620
   p90    : 14,249

At K=50 per cycle, this candidate pool is ~263 review cycles
-> prioritisation is the problem; the pool is far larger than capacity.


**Reading the printed output** (fill the blanks from the cells above when you run this — the interpretation, not the numbers, is what I'm committing to here):

1. **Inventory size.** A few tens of thousands of pages across a few dozen clients is a real if modest inventory — enough to build and validate a ranking, and enough distinct clients to hold some out entirely (see the split note below).

2. **Base rate — the number that decides whether this project is worth seven weeks.** Precision@50 is meaningless without it. If the share of `down` pages is small, then a 0.74 model is a genuine discovery; if it is large, a random draw already scores close to 0.74 and the model's real contribution shrinks to a handful of pages per cycle. **This is the number I check before anything else**, and it goes in the report next to every precision figure.

3. **Demand filter.** Restricting to pages that are both declining *and* still visible (≥100 impressions/90d) shows the `down` pages aren't just empty low-traffic noise — a meaningful subset would repay a reviewer's time. The candidate pool being many times larger than one cycle's capacity is precisely why *ordering* is the problem worth solving.

**Beginner trap I'm flagging for myself:** `trend_direction` and `trend_pct` are the exact columns the label (`is_declining_label`) is built from. I read them here only to describe the dataset — I will **never** use them as model features, or the model would just be copying its own answer (label leakage). Notebook 02 in the repo demonstrates exactly what that looks like.

**Split note:** evaluation must hold out ~20% of *clients*, not rows (`03_train_model.py` does this). A random row split would let pages from the same site sit on both sides, and the model would score well by memorising that site rather than by learning anything transferable. One consequence: the base rate must be recomputed **on the held-out clients**, since it can differ from the whole-file figure printed above.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I CAN say, if the analysis holds up:**
- "We *observed* that pages with X characteristics were more likely to carry the declining label in this 90-day window."
- "The ranked queue is *directional* — it prioritises review; it doesn't guarantee an outcome."
- "This is a *decision-support* tool: it orders candidates for a human reviewer; it does not publish or edit content itself."
- "On this starter slice, a learned model beat the hand-written baseline at Precision@50 on the same client-held-out split — reported next to the base rate, so the lift can be read honestly."

**What I will NOT say, ever:**
- That I *predicted* or *reverse-engineered* Google's ranking algorithm.
- That refreshing a page *caused* a recovery — there is no experiment here (no A/B test, no randomised holdout of refreshed vs. non-refreshed pages), only observational data.
- That `trend_direction == "down"` is proof of a *real* decline rather than seasonality, consolidation (a sibling page absorbing traffic), or plain noise — I have not ruled those out.
- That a high Precision@50 is impressive on its own, without the base rate beside it.
- That any hashed `client_id` / `content_id` maps back to a real client, domain, or URL.

**The limit underneath all of these:** the target is a rule-defined proxy (§2.4). Every number in this project measures agreement with `trend_direction`, not with "this page deserved review". A model that matched the proxy perfectly would still inherit every one of the proxy's mistakes — so "beat the baseline" means "ranked the proxy better", and that is the strongest claim available until an outcome measured in a later window replaces it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] **The notebook was actually run: `Runtime → Run all`, no errors, and the outputs are saved in the committed `.ipynb`** (an unexecuted notebook has no real numbers in it)
- [ ] The base rate is printed, and every precision figure in my write-up sits next to it
- [ ] All four framing questions are answered in writing: decision · who acts · cost of a wrong call · why ML over a plain rule
- [ ] The metric is named *before* any training, and the task type (ranking, via a classifier's probability) is stated
- [ ] The target is identified as a rule-defined proxy, with the gap described in both directions
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.